# AdaptiveIQ
### A learning system that actually pays attention to you

---

## Why I Built This

I have always felt that the problem with online learning isn't the content it's that no one is checking on you. You can be completely lost and the video just keeps playing. You can be bored out of your mind and the quiz still gives you the same easy questions. I wanted to build something that actually notices how you're doing and changes what it does based on that.

---

## What It Does

You type in any topic you're studying. AdaptiveIQ asks you questions about it but it's not just checking your answers. It's paying attention to how you answer. Are you uncertain? Taking a long time? Breezing through? Based on that, it decides what to do next. Maybe you need a simpler explanation. Maybe you need encouragement. Maybe you're ready for something harder. It adjusts every single round.

---

## Architecture

Three AI agents work together as a team:

 **Mood Detector**  analyzes your answer and response time to detect how you're feeling right now (confused / focused / bored / frustrated)
 **Decision Maker**  takes that mood and decides the best next action (simplify / encourage / increase difficulty / suggest break)
 **Explainer**  delivers a response shaped around your current state, not a generic answer

All session data is tracked in an MCP-inspired memory layer that remembers your moods, actions, and rounds across the entire session.

---

## How to Run It

1. Open the notebook on Kaggle
2. Add your Gemini API key in Kaggle Secrets as "GEMINI_API_KEY"
3. Run all cells in order
4. Type any topic and start learning!

>  Note: Uses Gemini free tier. If you hit a 503 or quota error, wait 1–2 minutes and run again. The system retries automatically.

---

## Built With

| Tool | Purpose |
|------|---------|
| Google Gemini 2.5 Flash | AI language model powering all three agents |
| Python | Core programming language |
| Kaggle Notebooks | Development and deployment environment |
| google-generativeai SDK | Gemini API integration |

---

## Agent Concepts Used

| Concept | How |
|---------|-----|
| Multi-agent system | Three agents working as a team in code |
| Session state tracking | MCP-inspired memory layer remembers what happened |
| Safety filtering | Blocks inappropriate topics before any agent runs |
| Deployability | Runs on Kaggle with a public shareable link |

In [1]:
from kaggle_secrets import UserSecretsClient
from google import genai

secret = UserSecretsClient()
api_key = secret.get_secret("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say ready in one word."
)
print(response.text)

Ready.


In [2]:

# AdaptIQ: Smart Learning Agent
# 3 agents working together to help students


from kaggle_secrets import UserSecretsClient
from google import genai
import time

# get API key safely
secret = UserSecretsClient()
api_key = secret.get_secret("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# small pause between calls to avoid hitting limits
def safe_generate(prompt):
    time.sleep(8)  # increase from 3 to 8 seconds
    result = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return result.text

# ── Agent 1: figures out how the student is feeling ──
def check_student_mood(student_input, response_time_seconds):
    prompt = f"""
    A student is learning and gave this response: "{student_input}"
    They took {response_time_seconds} seconds to answer.

    Based on this, decide how the student is feeling right now.
    Choose only one: confused / focused / bored / frustrated

    Reply in this exact format:
    State: [one word]
    Reason: [one short sentence]
    """
    return safe_generate(prompt)

# ── Agent 2: decides what to do next based on mood ──
def decide_next_step(mood, topic):
    prompt = f"""
    A student learning about "{topic}" is currently feeling: {mood}

    Decide what the teacher should do next.
    Choose one action: simplify / encourage / increase_difficulty / suggest_break

    Reply in this exact format:
    Action: [one word]
    Message: [one warm, friendly sentence to say to the student]
    """
    return safe_generate(prompt)

# ── Agent 3: explains the topic based on the action ──
def explain_topic(topic, action):
    prompt = f"""
    A student is learning about "{topic}".
    The recommended action is: {action}

    Write a short, friendly explanation of "{topic}" that fits the action.
    Keep it under 5 sentences. Use simple words. Be encouraging.
    """
    return safe_generate(prompt)

print("AdaptIQ agents are ready!")

AdaptIQ agents are ready!


In [3]:
# ── Safety Filter: blocks harmful topics ──
def is_safe_topic(topic):
    blocked = ["weapon", "drug", "hack", "kill", "violence", 
               "bomb", "illegal", "suicide", "adult", "porn"]
    topic_lower = topic.lower()
    for word in blocked:
        if word in topic_lower:
            print(f"\n AdaptiveIQ cannot help with that topic.")
            print("Please enter an appropriate academic subject.\n")
            return False
    return True

# ── Session Saver: stores session in memory ──
session_history = []

def save_session(topic, session_log):
    session_history.append({
        "topic": topic,
        "rounds": session_log
    })

# ── History Display: shows past sessions ──
def show_history():
    if not session_history:
        print("No previous sessions yet.")
        return
    print("\n Your Learning History:")
    print("=" * 50)
    for i, session in enumerate(session_history, 1):
        print(f"\n Session {i}: {session['topic']}")
        for entry in session["rounds"]:
            state = "engaged"
            for line in entry["mood"].split("\n"):
                if line.startswith("State:"):
                    state = line.replace("State:", "").strip()
            print(f"   Round {entry['round']}: Felt {state} → {entry['action']}")

In [4]:
# Main session: puts it all together
def run_adaptiveiq():
    print("=" * 50)
    print("       Welcome to AdaptiveIQ ")
    print("  Your personal smart learning system")
    print("=" * 50)

    topic = input("\nWhat topic are you studying today? ")

    # safety check first
    if not is_safe_topic(topic):
        return

    print(f"\nGreat! Let's learn about {topic} together.\n")

    session_log = []

    for round_num in range(1, 4):
        print(f"\n--- Round {round_num} ---")

        question = safe_generate(
            f"Ask one simple question about {topic} for a student. Just the question, nothing else."
        )
        print(f"\n Question: {question}")

        start = time.time()
        student_answer = input("Your answer: ")
        response_time = round(time.time() - start)

        print("\n Checking how you're feeling...")
        mood = check_student_mood(student_answer, response_time)
        print(mood)

        print("\n Adapting to you...")
        next_step = decide_next_step(mood, topic)
        print(next_step)

        action_word = "simplify"
        for line in next_step.split("\n"):
            if line.startswith("Action:"):
                action_word = line.replace("Action:", "").strip()

        print("\n Here's what I want you to know...")
        explanation = explain_topic(topic, action_word)
        print(explanation)

        session_log.append({
            "round": round_num,
            "answer": student_answer,
            "mood": mood,
            "action": action_word
        })

        print("\n" + "-" * 50)

    save_session(topic, session_log)

    print("\n Session Complete!")
    print(f"You completed 3 rounds on: {topic}")
    print("\nYour session summary:\n")

    for entry in session_log:
        state = "engaged"
        for line in entry['mood'].split("\n"):
            if line.startswith("State:"):
                state = line.replace("State:", "").strip()
        print(f"  Round {entry['round']}: Felt {state} → AdaptiveIQ chose to {entry['action']}")

    print("\nKeep learning, you're doing amazing! ")
    print("\n")
    show_history()

run_adaptiveiq()

       Welcome to AdaptiveIQ 
  Your personal smart learning system



What topic are you studying today?  computing ethics



Great! Let's learn about computing ethics together.


--- Round 1 ---

 Question: Is it ethically acceptable for a company to collect your personal data without directly asking, even if they claim it improves their services?


Your answer:  it's not ethical for the companies to collect personal data without person's knowledge but i am not sure about it



 Checking how you're feeling...
State: confused
Reason: The student explicitly stated "i am not sure about it" after taking a long time to formulate their response.

 Adapting to you...
Action: simplify
Message: No problem at all, let's try breaking this down into some smaller, simpler parts together.

 Here's what I want you to know...
Hi there! Computing ethics is all about using computers and technology in a good and fair way. It means thinking about what's right and wrong when you're online or using software. For example, it's ethical to respect other people's privacy and not copy someone else's work. By being mindful, you help make technology safe and helpful for everyone. You're doing great by learning about this important topic!

--------------------------------------------------

--- Round 2 ---

 Question: Should technology companies prioritize user privacy over profit?


Your answer:  definitely companies should consider user's privacy at every cost



 Checking how you're feeling...
State: Focused
Reason: The student took a moderate amount of time to articulate a clear, strong, and well-formed opinion, indicating thoughtful engagement with the topic.

 Adapting to you...
Action: increase_difficulty
Message: That was a very clear and well-articulated opinion – you're clearly thinking deeply about these ethics!

 Here's what I want you to know...
Computing ethics is all about making good, fair choices when we design and use technology. It helps us think about how computers affect people and society, covering important ideas like privacy, safety, and being responsible. You're doing a fantastic job grasping these concepts! Get ready to explore even deeper and tackle some exciting new ideas. You've got this!

--------------------------------------------------

--- Round 3 ---

 Question: Is it ever okay to use a computer to intentionally mislead or trick someone?


Your answer:  No, it's not okay 



 Checking how you're feeling...
State: Frustrated
Reason: The long time taken for a short, definitive negative answer suggests a struggle to find an acceptable solution.

 Adapting to you...
Action: encourage
Message: It sounds like you're really wrestling with some tough ideas, and that dedication to exploring all the angles is a true mark of thoughtful ethical thinking!

 Here's what I want you to know...
That's wonderful! Computing ethics is all about making good choices when we use computers and the internet. It helps us think about fairness, respecting privacy, and being kind to others online. Learning this means you're building awesome skills to use technology responsibly and make a positive impact. You're doing great by exploring these important ideas – keep up the fantastic work!

--------------------------------------------------

 Session Complete!
You completed 3 rounds on: computing ethics

Your session summary:

  Round 1: Felt confused → AdaptiveIQ chose to simplify
  Ro